<a href="https://colab.research.google.com/github/tommypolpo/geron-hands_on_ML/blob/main/ch12_ex8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8 - Build a CNN to achieve highest possible accuracy on MNIST


In [1]:
from sklearn.datasets import fetch_openml
import numpy as np
mnist = fetch_openml('mnist_784', as_frame = False)

X,y = mnist.data, mnist.target


In [2]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape, y_test.shape

((56000, 784), (56000,), (7000, 784), (7000,), (7000, 784), (7000,))

In [3]:
X_train[0].shape

(784,)

In [6]:
# reshape the images
X_train = X_train.reshape(56000, 28,28)
X_val = X_val.reshape(7000, 28,28)
X_test = X_test.reshape(7000, 28,28)
X_train.shape

(56000, 28, 28)

In [8]:
# transform to tensors
import torch
X_train = torch.tensor(X_train, dtype=torch.float32)/255
X_val = torch.tensor(X_val, dtype=torch.float32)/255
X_test = torch.tensor(X_test, dtype=torch.float32)/255


/tmp/ipykernel_683/65021492.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train, dtype=torch.float32)/255
/tmp/ipykernel_683/65021492.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_val = torch.tensor(X_val, dtype=torch.float32)/255
/tmp/ipykernel_683/65021492.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test, dtype=torch.float32)/255


In [10]:
# let's add the channel dimension before the spatial dimensions
X_train = X_train.reshape(56000, 1, 28, 28)
X_val = X_val.reshape(7000, 1, 28,28)
X_test = X_test.reshape(7000, 1, 28,28)
X_train.shape

torch.Size([56000, 1, 28, 28])

### Let's build an inception module

In [ ]:
from torch.nn import nn
from functools import partial
import torch.nn.functional as F

class InceptionModule(nn.Module):
  def __init__(self, in_channels):
    super().__init__()
    DeafultConv2d = partial(nn.Conv2d,
                            in_channels=in_channels,
                            out_channels=in_channels,
                            stride=1, padding="same", bias=False
                            )
    self.conv1x1 = DeafultConv2d(kernel_size=1)
    self.conv3x3 = DeafultConv2d(kernel_size=3)
    self.conv5x5 = DeafultConv2d(kernel_size=5)
    self.maxpool = nn.MaxPool2d(kernel_size=3, stride=1, padding="same")
    self.conv1_3 = nn.Sequential(
        self.conv1x1,
        nn.ReLU(),
        self.conv3x3,
        nn.ReLU()
    )
    self.conv1_5 = nn.Sequential(
        self.conv1x1,
        nn.ReLU(),
        self.conv5x5,
        nn.ReLU()
    )
    self.maxpool_conv1 = nn.Sequential(
        self.maxpool,
        self.conv1x1,
        nn.ReLU()
    )
    self.depth_concat = torch.cat((
        self.conv1_1,
        self.conv1_3,
        self.conv1_5,
        self.maxpool_conv1), dim=1)

  def forward(self, inputs):
    return F.relu(self.depth_concat)